# 📄 Chat with a PDF — RAG Pipeline from Scratch

**Retrieval-Augmented Generation (RAG)** is one of the most common real-world AI engineering tasks.  
This notebook builds a complete RAG pipeline from scratch — no magic, no black boxes.

### What we build
Given any PDF, we can ask it questions and get grounded answers.

### The pipeline (6 steps)
```
PDF URL → Load → Chunk → Embed → FAISS Index → Retrieve → Groq LLM → Answer
```

### Stack
| Component | Tool | Why |
|---|---|---|
| PDF loading | `pypdf` | Extracts raw text from any PDF |
| Chunking | `langchain-text-splitters` | Splits text into overlapping windows |
| Embeddings | `sentence-transformers` (HuggingFace) | Free, runs locally, no API key needed |
| Vector DB | `FAISS` | Fast similarity search by Facebook AI |
| LLM | `Groq` (llama-3.3-70b) | Free API, extremely fast inference |

---
**Paper used:** [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) — Vaswani et al., 2017

## Step 1 — Install Dependencies

In [ ]:
# Install all required packages
# - langchain + langchain-community: framework that connects all components
# - langchain-huggingface: lets LangChain use HuggingFace embedding models
# - langchain-text-splitters: text chunking utilities
# - faiss-cpu: Facebook's vector similarity search library
# - sentence-transformers: the actual embedding model (all-MiniLM-L6-v2)
# - pypdf: PDF text extraction
# - groq: Groq API client for the LLM

!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    groq

print("All packages installed!")

## Step 2 — Load the PDF

We download the PDF into memory and extract raw text from all pages.  
No file is saved to disk — we read it directly from the URL.

In [ ]:
import urllib.request
from pypdf import PdfReader
import io

# --- Config: swap this URL to use any other PDF ---
PDF_URL = "https://arxiv.org/pdf/1706.03762"

# Download the PDF bytes into memory
req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
pdf_bytes = urllib.request.urlopen(req).read()

# Extract text from every page
reader = PdfReader(io.BytesIO(pdf_bytes))
pages = [page.extract_text() for page in reader.pages]
full_text = "\n".join(pages)

print(f"Pages loaded   : {len(pages)}")
print(f"Total chars    : {len(full_text):,}")
print(f"\n--- First 400 characters ---")
print(full_text[:400])

## Step 3 — Chunk the Text

LLMs have a context limit — we can't feed the whole PDF at once.  
We split it into small overlapping chunks so no sentence is cut off at a boundary.

**Key parameters:**
- `chunk_size=500` — max characters per chunk
- `chunk_overlap=50` — how many characters to repeat between adjacent chunks

The overlap ensures a sentence split across two chunks isn't lost.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # max characters per chunk
    chunk_overlap=50,      # overlap to preserve context at boundaries
    separators=["\n\n", "\n", " ", ""]  # tries paragraph breaks first
)

chunks = splitter.split_text(full_text)

print(f"Total chunks   : {len(chunks)}")
print(f"Avg chunk size : {sum(len(c) for c in chunks) // len(chunks)} characters")

# Show the overlap in action
print(f"\n--- End of chunk 0 ---")
print(repr(chunks[0][-80:]))
print(f"\n--- Start of chunk 1 ---")
print(repr(chunks[1][:80]))
print("\n↑ Same text appears at the end of chunk 0 and start of chunk 1 — that's the overlap.")

## Step 4 — Embed the Chunks

We convert each chunk into a **vector** (a list of numbers) that represents its meaning.  
Chunks with similar meaning will have similar vectors — this is what enables semantic search.

**Model:** `all-MiniLM-L6-v2` — a small, fast HuggingFace model that produces 384-dimensional vectors.  
Downloads ~90MB on first run, then cached.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load the embedding model (downloads on first run, cached after)
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Embed a single chunk to inspect what a vector looks like
sample_vector = embedder.embed_query(chunks[0])

print(f"Embedding model : all-MiniLM-L6-v2")
print(f"Vector size     : {len(sample_vector)} dimensions")
print(f"\nFirst 10 values of chunk 0's vector:")
print([round(v, 4) for v in sample_vector[:10]])
print("\nThose 384 numbers encode the *meaning* of the text.")
print("Similar text → similar numbers → found by similarity search.")

## Step 5 — Build the FAISS Vector Index

FAISS (Facebook AI Similarity Search) stores all 89 vectors and lets us search them instantly.  
When we embed a query, FAISS finds the chunks whose vectors are closest to the query vector.

This is **cosine similarity** — not keyword matching.

In [ ]:
from langchain_community.vectorstores import FAISS

# Embed all chunks and build the searchable index
print("Building FAISS index...")
db = FAISS.from_texts(chunks, embedder)
print(f"Index built! Vectors stored: {db.index.ntotal}")

# --- Test retrieval immediately ---
test_query = "What is the attention mechanism?"
test_results = db.similarity_search(test_query, k=3)

print(f"\n--- Test: top 3 chunks for '{test_query}' ---")
for i, doc in enumerate(test_results):
    print(f"\n[{i+1}] {doc.page_content[:200]}...")

## Step 6 — Load Groq API Key

We load the key securely from Kaggle Secrets — never hardcode API keys in notebooks.

**To add your key:**  
Right panel → Add-ons → Secrets → Add `GROQ_API_KEY` → toggle on.  
Get a free key at [console.groq.com](https://console.groq.com).

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")

print("Groq API key loaded successfully!")

## Step 7 — Ask the PDF a Question

The full RAG loop in one cell:
1. Embed the query
2. Retrieve top-k most relevant chunks from FAISS
3. Build a prompt with those chunks as context
4. Send to Groq LLM
5. Return grounded answer

**Change `query` to anything and re-run.**

In [ ]:
from groq import Groq

def ask_pdf(query, k=5):
    """
    Full RAG pipeline:
      query -> retrieve top-k chunks -> build prompt -> LLM -> answer
    
    Args:
        query: the question to ask
        k: number of chunks to retrieve (more = richer context, higher token cost)
    """
    # Step 1: retrieve relevant chunks via vector similarity
    results = db.similarity_search(query, k=k)
    context = "\n\n".join(doc.page_content for doc in results)

    # Step 2: build a grounded prompt
    # "ONLY using the context" is critical — prevents hallucination
    system_prompt = (
        "You are a helpful assistant that answers questions about a research paper.\n"
        "Answer ONLY using the context provided below. "
        "If the answer is not in the context, say 'I don\'t find that in the paper.' "
        "Do not make anything up.\n\n"
        f"Context:\n{context}"
    )

    # Step 3: send to Groq
    client = Groq(api_key=GROQ_API_KEY)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query}
        ]
    )

    answer = response.choices[0].message.content

    # Print everything for full transparency
    print("=" * 60)
    print(f"QUERY  : {query}")
    print("=" * 60)
    print(f"CONTEXT (top {k} chunks, first 600 chars):")
    print(context[:600], "...")
    print("=" * 60)
    print("ANSWER :")
    print(answer)
    print("=" * 60)
    return answer


# --- Try different questions below ---
ask_pdf("How does multi-head attention work?")

In [ ]:
# Try more questions — change and re-run this cell
ask_pdf("What BLEU score did the Transformer achieve on WMT 2014 English-to-German?")

In [ ]:
ask_pdf("What optimizer and learning rate schedule did they use?")

In [ ]:
ask_pdf("Why did the authors remove recurrence from the architecture?")

## What We Learned

### RAG works because of semantic search
When we search for *"side effects"*, it also finds chunks about *"adverse reactions"* — same meaning, different words. Keyword search would miss that.

### Retrieval quality controls answer quality
The LLM can only answer as well as what we give it. If the right chunk isn't retrieved, the answer will be weak or "I don't find that." This is expected and honest behavior — not a bug.

### Tuning levers
| Parameter | Effect |
|---|---|
| `chunk_size` | Smaller = more precise retrieval. Larger = more context per chunk. |
| `chunk_overlap` | Higher = fewer boundary misses, more storage. |
| `k` (top-k) | Higher = richer context, but more tokens sent to LLM. |
| `model` | Larger models = better reasoning over complex context. |

### Common interview follow-ups
- *How would you evaluate retrieval quality?* → RAGAS, recall@k, MRR
- *What if the answer spans multiple chunks?* → Increase k, or use a re-ranker (Cohere Rerank)
- *How would you deploy this?* → FastAPI endpoint, FAISS index saved to disk, reload at startup
- *How do you handle PDFs with images/tables?* → OCR (pytesseract), or multimodal models

---
*Built with LangChain · HuggingFace · FAISS · Groq*